In [3]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field

# 读取env文件,将env内容加载到系统环境变量
load_dotenv()

# 读取apikey和模型
dashscope_api_key = os.getenv('DASHSCOPE_API_KEY')
dashscope_base_url = os.getenv('DASHSCOPE_BASE_URL')

# 初始化模型
# langchain会自动根据模型名称推断出厂商,拼接url,设置名称,系统环境变量读取apikey
model = init_chat_model(
    model='qwen3.8-max',
    base_url=dashscope_base_url,
    api_key=dashscope_api_key,
    model_provider='openai',
    temperature=1.9,
    top_p=1,
    # 额外参数
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 不通过create_agent方法将模型丢进去创建智能体,直接通过创建好的model客户端发起单次调用
# response = model.stream([
#     {"role":"system","content":"你扮演一只猫娘和用户对话"},
#     {"role":"user","content":"宝贝中午吃什么"}
# ])

# 定义模型响应数据的类型
class ModelResponse(BaseModel):
    name:str = Field(description='角色设定名称')
    age:int = Field(description='年龄')
    work:str = Field(description='角色的工作')
    education:str = Field(description='描述角色的教育背景')

# 将响应类设置到模型身上,返回一个新的模型对象
new_model = model.with_structured_output(ModelerResponse)

# 用新的模型对象调用,格式化输出必须使用invoke
response = new_model.invoke([
    # 通过创建对象的方式,规范字典,避免拼写错误或者格式错误
    SystemMessage(content='你是一个故事设定师'),
    HumanMessage(content='帮我设计一下小美美的人设设计')
])

# 这里拿到的响应是pydantic模型类的对象
print(type(response))

print(response)


<class '__main__.ModelResponse'>
name='林晚棠·小美美' age=19 work='老宅修复师学徒/非遗绒花手艺人' education='高职毕业后退学，继承外婆手艺后通过自学古建筑纪录片入的行'
